# CrackFM on Kaggle

End-to-end run on Kaggle's free GPU (2×T4). PyTorch is preinstalled on Kaggle GPU images; this notebook installs CrackFM's light deps, trains the U-Net baseline, and reports metrics + severity.

1. Add a crack dataset to the notebook (e.g. a CRACK500/DeepCrack Kaggle dataset) so it appears under `/kaggle/input/...`.
2. Point `DATA_ROOT` at a folder containing `images/` and `masks/` (or adapt the cell below to build that layout).

In [ ]:
# Clone CrackFM and install light deps (PyTorch already present on Kaggle GPU).
!git clone https://github.com/anushkasinghania/CrackFM.git
%cd CrackFM
!pip install -q scikit-image opencv-python-headless omegaconf albumentations 2>/dev/null
import sys; sys.path.insert(0, 'src')

In [ ]:
# Sanity: pure-numpy core tests (no GPU needed).
!PYTHONPATH=src python -m pytest tests/test_metrics.py tests/test_prompting.py tests/test_severity.py -q

In [ ]:
# Set this to a folder with images/ and masks/ subfolders.
DATA_ROOT = '/kaggle/input/your-crack-dataset'  # <-- edit me

# Train the U-Net baseline end-to-end.
!PYTHONPATH=src python -m crackfm.train \
    data.root=$DATA_ROOT model.name=unet \
    data.image_size=448 data.batch_size=8 trainer.epochs=30 \
    tracking.backend=none trainer.out_dir=runs

In [ ]:
# Evaluate + severity report.
!PYTHONPATH=src python -m crackfm.eval ckpt=runs/best.pt data.root=$DATA_ROOT

## Geometry-adaptive prompting demo
Visualise the prompts CrackFM derives from a coarse mask (the core contribution).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from crackfm.prompting import GeometryAdaptivePrompter

m = np.zeros((128,128), bool)
for i in range(20,100): m[i, i//2+20:i//2+22] = True  # a curved-ish crack
m[60:80, 70:72] = True                                 # a branch
ps = GeometryAdaptivePrompter(n_points=24, neg_points=8)(m)

plt.imshow(m, cmap='gray')
pos = ps.point_labels==1
plt.scatter(ps.point_coords[pos,0], ps.point_coords[pos,1], c='lime', s=18, label='pos')
plt.scatter(ps.point_coords[~pos,0], ps.point_coords[~pos,1], c='red', s=18, label='neg')
for x0,y0,x1,y1 in ps.boxes:
    plt.gca().add_patch(plt.Rectangle((x0,y0), x1-x0, y1-y0, fill=False, ec='cyan'))
plt.legend(); plt.title('Geometry-adaptive prompts'); plt.show()